# FloodGuard AI — full training run on Colab / Kaggle
Ensemble learning framework for urban flood risk prediction (Kaggle Playground S4E5).

**Runtime → Change runtime type → GPU (T4).** XGBoost, CatBoost and the PyTorch MLP use the GPU automatically (`models.use_gpu: auto`); LightGBM can use the GPU with `--override models.lgbm_device=gpu` (falls back to CPU if the installed build lacks GPU support). A full run (40 Optuna trials × 7 models on ~1.1 M rows) takes several hours; lower `optuna.n_trials` or set `data.sample_rows` for a shorter run.

In [ ]:
!nvidia-smi || echo 'No GPU - the run will be slow'

## 1. Get the code and install dependencies

In [ ]:
REPO_URL = 'https://github.com/<your-user>/FloodGuard-AI.git'  # <- your GitHub repository
!git clone -q $REPO_URL FloodGuard-AI
%cd FloodGuard-AI
!pip install -q -r requirements.txt && pip install -q -e .

## 2. Kaggle credentials
Upload `kaggle.json` (Kaggle → Settings → API → *Create New Token*) and accept the [competition rules](https://www.kaggle.com/competitions/playground-series-s4e5) once in the browser.

In [ ]:
import os
from google.colab import files
files.upload()  # choose kaggle.json
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!python scripts/run_pipeline.py --stages download

## 3. (Optional) 3-minute smoke test on synthetic data

In [ ]:
!python scripts/make_synthetic_flood.py && python scripts/run_pipeline.py --override configs/smoke.yaml --stages all --skip download

## 4. Full pipeline
Stages: prepare → features → reduce → tune → train → ensemble → evaluate → benchmark → explain → plots → report. Optuna studies are stored in `runs/optuna/optuna.db`, so an interrupted `tune` stage resumes where it stopped.

In [ ]:
# Shorter run: add  --override optuna.n_trials=15 --override data.sample_rows=400000
!python scripts/run_pipeline.py --stages all --skip download

## 5. Results

In [ ]:
import json, pandas as pd
ev = json.load(open('runs/metrics/evaluation.json'))
rows = [{'model': r['name'], 'OOF R2': r['oof_r2'], 'test R2': r['test_r2']['value'],
         'CI low': r['test_r2']['lo'], 'CI high': r['test_r2']['hi'], 'macro-F1': r.get('test_macro_f1'),
         'ROC-AUC': r.get('test_roc_auc'), 'selected': r['selected']} for r in ev['models']]
pd.DataFrame(rows).sort_values('test R2', ascending=False).round(5)

In [ ]:
print(json.dumps(json.load(open('runs/metrics/statistical_tests.json')), indent=1)[:2500])

In [ ]:
from IPython.display import Image, display
import glob
for path in sorted(glob.glob('runs/figures/*.png')):
    print(path)
    display(Image(path, width=900))

## 6. MLflow
Download `runs/mlflow.db` and run locally:

```
mlflow ui --backend-store-uri sqlite:///runs/mlflow.db
```

## 7. Leaderboard benchmark
Submit `runs/submission.csv` as a late submission on Kaggle to obtain the private-leaderboard R² of the ensemble, and copy the private top/median scores into `configs/benchmarks.yaml` (never guess them). Then re-run `--stages benchmark,report`.

In [ ]:
!python scripts/run_pipeline.py --stages benchmark,report

## 7b. Report for Overleaf
Creates `report_overleaf.zip` (tex + tables + figures). On overleaf.com: **New Project → Upload Project →** choose the zip → **Recompile** (pdfLaTeX). Then *Download PDF*.

In [ ]:
!python scripts/package_report.py
files.download('report_overleaf.zip')

## 8. Save outputs

In [ ]:
!zip -qr floodguard_outputs.zip runs/models runs/metrics runs/figures runs/optuna runs/mlflow.db runs/submission.csv report/tables report/figures
files.download('floodguard_outputs.zip')

## 9. (Optional) Deploy to Hugging Face Spaces
Add `HF_TOKEN` (write) in Colab *Secrets*, then:

In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_USER'] = '<your-hf-username>'
!bash deploy/push_to_hf_space.sh